# EDA PADRÓN MUNICIPAL - Análisis de datos demográficos

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

### Carga y exploración de estructura

In [2]:
df = pd.read_parquet("../data/processed/01_consolidated/dataset_padron.parquet")

print(f"\n ESTRUCTURA PADRÓN")
print(f"  Filas: {len(df):,}")
print(f"  Columnas: {df.shape[1]}")
print(f"  Tamaño memoria: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB")
df.head()



 ESTRUCTURA PADRÓN
  Filas: 241,125
  Columnas: 15
  Tamaño memoria: 52.09 MB


,COD_DISTRITO,DESC_DISTRITO,COD_DIST_BARRIO,DESC_BARRIO,COD_BARRIO,COD_DIST_SECCION,COD_SECCION,COD_EDAD_INT,ESPANOLESHOMBRES,ESPANOLESMUJERES,EXTRANJEROSHOMBRES,EXTRANJEROSMUJERES,FX_CARGA,FX_DATOS_INI,FX_DATOS_FIN
0,2,ARGANZUELA,202,ACACIAS,2,2019.0,19.0,81,3,9,0,0,2026-07-14 08:28:59.546393,2026-07-01,2026-07-01
1,8,FUENCARRAL-EL PARDO,805,LA PAZ,5,8098.0,98.0,74,11,10,0,0,2026-07-14 08:28:59.546393,2026-07-01,2026-07-01
2,1,CENTRO,105,UNIVERSIDAD,5,1092.0,92.0,81,2,3,1,0,2026-07-14 08:28:59.546393,2026-07-01,2026-07-01
3,2,ARGANZUELA,201,IMPERIAL,1,2009.0,9.0,92,2,7,0,0,2026-07-14 08:28:59.546393,2026-07-01,2026-07-01
4,5,CHAMARTIN,505,NUEVA ESPAÑA,5,5084.0,84.0,74,9,9,1,0,2026-07-14 08:28:59.546393,2026-07-01,2026-07-01


### Análisis de nulos

In [3]:

nulos = df.isnull().sum()   
if nulos.sum() == 0:
    print(f"\n Sin valores nulos")
    
print(f"\n  Valores nulos por columna:")
nulos_df = pd.DataFrame({
        'Columna': nulos.index,
        'Nulos': nulos.values,
        '%': (nulos.values / len(df) * 100).round(2)
    })
nulos_df = nulos_df[nulos_df['Nulos'] > 0].sort_values('Nulos', ascending=False)
    
for _, row in nulos_df.iterrows():
        print(f"  {row['Columna']:30s} {row['Nulos']:>8,d}  ({row['%']:>6.2f}%)")


  Valores nulos por columna:
  COD_DIST_SECCION                      1  (  0.00%)
  COD_SECCION                           1  (  0.00%)


### Análisis de duplicados

In [4]:
duplicados_totales = df.duplicated().sum()
print(f"  Total (todas las columnas): {duplicados_totales:,}")

  Total (todas las columnas): 0


### Análisis de tipos de datos

In [5]:
print(f"\n Tipos de datos:")
tipos = df.dtypes.value_counts()
for tipo, cantidad in tipos.items():
    print(f"  {str(tipo):25s} {cantidad:3d} columnas")

df.info()


 Tipos de datos:
  int64                       7 columnas
  str                         6 columnas
  float64                     2 columnas
<class 'pandas.DataFrame'>
RangeIndex: 241125 entries, 0 to 241124
Data columns (total 15 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   COD_DISTRITO        241125 non-null  int64  
 1   DESC_DISTRITO       241125 non-null  str    
 2   COD_DIST_BARRIO     241125 non-null  int64  
 3   DESC_BARRIO         241125 non-null  str    
 4   COD_BARRIO          241125 non-null  int64  
 5   COD_DIST_SECCION    241124 non-null  float64
 6   COD_SECCION         241124 non-null  float64
 7   COD_EDAD_INT        241125 non-null  str    
 8   ESPANOLESHOMBRES    241125 non-null  int64  
 9   ESPANOLESMUJERES    241125 non-null  int64  
 10  EXTRANJEROSHOMBRES  241125 non-null  int64  
 11  EXTRANJEROSMUJERES  241125 non-null  int64  
 12  FX_CARGA            241125 non-null  str    
 13  FX_D

### Análisis de cobertura geografica

In [6]:
print(f"\n  COBERTURA GEOGRÁFICA:")
if 'DESC_DISTRITO' in df.columns:
    distritos = df['DESC_DISTRITO'].nunique()
    print(f"  Distritos: {distritos}")
        
    print(f"\n  Distritos únicos:")
    for dist in sorted(df['DESC_DISTRITO'].unique()):
        count = len(df[df['DESC_DISTRITO'] == dist])
        print(f"    {dist:30s} {count:>10,} registros")
    
if 'DESC_BARRIO' in df.columns:
    barrios = df['DESC_BARRIO'].nunique()
    print(f"  Barrios: {barrios}")
    
if 'COD_EDAD_INT' in df.columns:
    edades = df['COD_EDAD_INT'].nunique()
    print(f"  Rangos de edad: {edades}")  



  COBERTURA GEOGRÁFICA:
  Distritos: 21

  Distritos únicos:
    ARGANZUELA                         10,865 registros
    BARAJAS                             3,111 registros
    CARABANCHEL                        17,849 registros
    CENTRO                             10,673 registros
    CHAMARTIN                          10,019 registros
    CHAMBERI                           11,995 registros
    CIUDAD LINEAL                      16,587 registros
    FUENCARRAL-EL PARDO                18,157 registros
    HORTALEZA                          12,234 registros
    LATINA                             19,261 registros
    MONCLOA-ARAVACA                     8,861 registros
    MORATALAZ                           8,158 registros
    PUENTE DE VALLECAS                 17,178 registros
    RETIRO                              9,212 registros
    SALAMANCA                          12,261 registros
    SAN BLAS-CANILLEJAS                10,963 registros
    TETUAN                             11,

### Análisis de población

In [7]:
print("\n POBLACIÓN")
if 'ESPANOLESHOMBRES' in df.columns and 'ESPANOLESMUJERES' in df.columns:
    df_temp = df.copy()
    df_temp['ESPAÑOLES_TOTAL'] = df_temp['ESPANOLESHOMBRES'] + df_temp['ESPANOLESMUJERES']
    poblacion_esp = df_temp['ESPAÑOLES_TOTAL'].sum()
    print(f"  Españoles (total): {poblacion_esp:>12,.0f}")
    
if 'EXTRANJEROSHOMBRES' in df.columns and 'EXTRANJEROSMUJERES' in df.columns:
    df_temp = df.copy()
    df_temp['EXTRANJEROS_TOTAL'] = df_temp['EXTRANJEROSHOMBRES'] + df_temp['EXTRANJEROSMUJERES']
    poblacion_ext = df_temp['EXTRANJEROS_TOTAL'].sum()
    print(f"  Extranjeros (total): {poblacion_ext:>12,.0f}")
        
    total = poblacion_esp + poblacion_ext if 'poblacion_esp' in locals() else poblacion_ext
    print(f"  Total (estimado): {total:>17,.0f}")


 POBLACIÓN
  Españoles (total):    2,823,087
  Extranjeros (total):      689,763
  Total (estimado):         3,512,850


### Estadísticas descriptivas

In [8]:
print(f"\n ESTADÍSTICAS NUMÉRICAS:")
    
numericas = df.select_dtypes(include=[np.number]).columns
if len(numericas) == 0:
    print("  No hay columnas numéricas")
        
    
for col in numericas[:5]:  # Top 5
    print(f"  {col}:")
    print(f"    Media: {df[col].mean():>12.0f}")
    print(f"    Min: {df[col].min():>13.0f}, Max: {df[col].max():>12.0f}")



 ESTADÍSTICAS NUMÉRICAS:
  COD_DISTRITO:
    Media:           10
    Min:             1, Max:           21
  COD_DIST_BARRIO:
    Media:         1046
    Min:           101, Max:         2105
  COD_BARRIO:
    Media:            4
    Min:             1, Max:            9
  COD_DIST_SECCION:
    Media:        10495
    Min:          1001, Max:        21034
  COD_SECCION:
    Media:           74
    Min:             1, Max:          222
